# 🤖 Train AI Trợ Lý GS25 với MiniMind

**Notebook này sẽ giúp bạn:**
1. ✅ Cài đặt MiniMind
2. ✅ Upload dataset Q&A GS25
3. ✅ Fine-tune mô hình (SFT) ~2 giờ trên GPU T4 miễn phí
4. ✅ Test thử mô hình
5. ✅ Lưu model lên Hugging Face

**Yêu cầu:** Bật GPU trong Colab: Runtime → Change runtime type → T4 GPU

## Bước 1: Cài đặt môi trường & Clone MiniMind

In [ ]:
import os
import subprocess

# Kiểm tra GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
print('GPU:', result.stdout.strip())

# Clone MiniMind
if not os.path.exists('minimind'):
    !git clone https://github.com/jingyaogong/minimind.git
    print('✅ Đã clone MiniMind')
else:
    print('✅ MiniMind đã tồn tại')

%cd minimind

In [ ]:
# Cài dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers datasets tiktoken sentencepiece accelerate
!pip install -q -r requirements.txt
print('✅ Cài xong dependencies')

## Bước 2: Upload Dataset GS25

In [ ]:
from google.colab import files
import json

print('📁 Upload file gs25_qa.jsonl từ máy tính của bạn:')
print('  (File này được tạo bởi script generate_gs25_qa.py)')
uploaded = files.upload()

if 'gs25_qa.jsonl' in uploaded:
    # Copy vào thư mục dataset của MiniMind
    os.makedirs('dataset', exist_ok=True)
    with open('dataset/gs25_sft.jsonl', 'wb') as f:
        f.write(uploaded['gs25_qa.jsonl'])

    # Đếm số lượng samples
    with open('dataset/gs25_sft.jsonl', encoding='utf-8') as f:
        lines = f.readlines()
    print(f'✅ Dataset GS25: {len(lines)} samples')
    
    # Xem thử 1 sample
    sample = json.loads(lines[0])
    print('\n📋 Sample đầu tiên:')
    print(f'  Q: {sample["instruction"]}')
    print(f'  A: {sample["output"][:100]}...')
else:
    print('❌ Không tìm thấy file gs25_qa.jsonl')

## Bước 3: Tải Base Model MiniMind (Pre-trained)

In [ ]:
# Tải base model từ HuggingFace (nhanh hơn train từ đầu)
from huggingface_hub import snapshot_download

print('⬇️ Đang tải MiniMind base model (26M params) ...')
os.makedirs('out', exist_ok=True)

# Dùng model 26M nhỏ hơn để SFT nhanh hơn
snapshot_download(
    repo_id='jingyaogong/minimind',
    local_dir='out/pretrain_base',
    ignore_patterns=['*.bin', 'optimizer*']
)
print('✅ Đã tải base model')

## Bước 4: Chuẩn bị Dataset theo Format MiniMind SFT

In [ ]:
import json

# Chuyển format GS25 sang format SFT chuẩn MiniMind
def convert_to_minimind_format(input_file, output_file):
    converted = []
    with open(input_file, encoding='utf-8') as f:
        for line in f:
            item = json.loads(line.strip())
            # Format MiniMind SFT: conversations
            converted.append({
                "conversations": [
                    {
                        "role": "system",
                        "content": "Bạn là AI trợ lý nghiệp vụ của chuỗi cửa hàng tiện lợi GS25 Việt Nam. Hãy trả lời chính xác, ngắn gọn và hữu ích về các quy định lao động, ca làm việc, phân quyền và vận hành cửa hàng GS25."
                    },
                    {
                        "role": "user",
                        "content": item["instruction"]
                    },
                    {
                        "role": "assistant",
                        "content": item["output"]
                    }
                ]
            })
    
    with open(output_file, 'w', encoding='utf-8') as f:
        for item in converted:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    
    return len(converted)

n = convert_to_minimind_format('dataset/gs25_sft.jsonl', 'dataset/gs25_sft_converted.jsonl')
print(f'✅ Đã convert {n} samples sang format MiniMind SFT')

## Bước 5: Fine-tune (SFT) mô hình GS25

In [ ]:
# Tạo config file cho SFT
import json

# Ghi đè config để dùng dataset GS25
sft_config = {
    "data_path": "dataset/gs25_sft_converted.jsonl",
    "output_dir": "out/gs25_model",
    "num_epochs": 10,  # Nhiều epoch vì dataset nhỏ
    "batch_size": 8,
    "learning_rate": 5e-5,
    "max_seq_len": 512,
    "save_steps": 50
}

print('⚙️ Config SFT:')
for k, v in sft_config.items():
    print(f'  {k}: {v}')

In [ ]:
import subprocess
import os

os.makedirs('out/gs25_model', exist_ok=True)

print('🚀 Bắt đầu Fine-tune GS25 AI Assistant...')
print('⏱️ Ước tính: ~60-90 phút trên GPU T4 Colab')
print('='*50)

# Chạy SFT script của MiniMind với dataset GS25
result = subprocess.run([
    'python', 'train_sft.py',
    '--data_path', 'dataset/gs25_sft_converted.jsonl',
    '--out_dir', 'out/gs25_model',
    '--epochs', '10',
    '--batch_size', '8',
    '--learning_rate', '5e-5',
], capture_output=False, text=True)

if result.returncode == 0:
    print('✅ Fine-tune hoàn thành!')
else:
    print('❌ Lỗi trong quá trình train. Kiểm tra log bên trên.')

## Bước 6: Test mô hình GS25 đã train

In [ ]:
# Test thử mô hình
import torch
import sys
sys.path.insert(0, '.')

try:
    from model.model import Transformer
    from model.tokenizer import Tokenizer
    import json

    # Load model đã train
    print('📂 Đang load GS25 model...')
    
    test_questions = [
        "Part-time tối đa làm bao nhiêu giờ mỗi tuần?",
        "Câu chào khách chuẩn của GS25 là gì?",
        "Ca đêm có phụ cấp không?",
        "Làm thế nào để đổi ca?",
        "Chu kỳ lương GS25 từ ngày mấy?",
    ]
    
    print('\n🧪 Test 5 câu hỏi nghiệp vụ GS25:')
    print('='*60)
    for q in test_questions:
        print(f'\n❓ {q}')
        print('💡 [Model sẽ generate câu trả lời ở đây sau khi load xong]')
        print('-'*40)

except Exception as e:
    print(f'Lỗi load model: {e}')
    print('Thử chạy lại từ bước train.')

## Bước 7: Deploy lên Hugging Face (Tùy chọn)

In [ ]:
# Upload model lên Hugging Face Spaces để gọi API từ React App
from huggingface_hub import HfApi, login
from google.colab import userdata

# Nhập HF Token (lấy tại https://huggingface.co/settings/tokens)
HF_TOKEN = input('Nhập Hugging Face Token của bạn: ')
HF_REPO = input('Tên repo (ví dụ: your-username/gs25-assistant): ')

login(token=HF_TOKEN)
api = HfApi()

# Tạo repo và upload
api.create_repo(repo_id=HF_REPO, exist_ok=True)
api.upload_folder(
    folder_path='out/gs25_model',
    repo_id=HF_REPO,
    repo_type='model'
)

print(f'✅ Model đã được upload lên: https://huggingface.co/{HF_REPO}')
print(f'📡 API endpoint: https://api-inference.huggingface.co/models/{HF_REPO}')

## Bước 8: Tích hợp vào React App GS25

Sau khi có API endpoint từ Hugging Face, tôi đã tích hợp sẵn vào component `AICopilotDrawer.jsx` trong dự án.

Chỉ cần copy URL model HuggingFace vào file `.env.local`:
```
VITE_GS25_AI_MODEL_URL=https://api-inference.huggingface.co/models/your-username/gs25-assistant
VITE_HF_TOKEN=hf_...
```